# 02 Baselines

A majority baseline and logistic regression on word and on character TF-IDF features, then the winning character model again with balanced class weights. Every choice is made on development macro-F1, and the test split is not read here.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Keaane/Ikinyamakuru/blob/main/notebooks/02_baselines.ipynb)

In [1]:
# Setup: run from the repository root, on a laptop or on Colab.
import os
import subprocess
import sys
import warnings
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/Ikinyamakuru").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Keaane/Ikinyamakuru.git", "/content/Ikinyamakuru"], check=True)
    os.chdir("/content/Ikinyamakuru")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path("src").resolve()))

# Show each warning as one short line.
warnings.showwarning = lambda message, category, *args, **kwargs: print(f"{category.__name__}: {message}")

In [2]:
import json
import time
import warnings
from collections import Counter

import joblib
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from inkuru.data import align_proba, load_split, records, write_jsonl
from inkuru.metrics import classification_scores
from inkuru.paths import BASELINES, RESULTS, SEED

train, dev = load_split("train"), load_split("dev")
train_text, train_y = [row["text"] for row in train], [row["label"] for row in train]
dev_text, dev_y = [row["text"] for row in dev], [row["label"] for row in dev]
BASELINES.mkdir(parents=True, exist_ok=True)
CS = [0.25, 1.0, 4.0, 16.0]
print(f"train {len(train):,} articles, development {len(dev):,}")

train 6,864 articles, development 763


## Majority baseline

It always predicts the most common training section, which shows what the class prior alone scores under macro-F1.

In [3]:
majority = Counter(train_y).most_common(1)[0][0]
prior = Counter(train_y)[majority] / len(train_y)
write_jsonl(RESULTS / "majority_dev.jsonl", [
    {"id": row["id"], "gold": row["label"], "pred": majority, "confidence": prior, "top": [{"label": majority, "p": prior}]}
    for row in dev
])
(BASELINES / "majority.json").write_text(json.dumps({"label": majority}) + "\n")
print(majority, "development macro-F1", round(classification_scores(dev_y, [majority] * len(dev))["macro_f1"], 4))

politics development macro-F1 0.0274


## Word and character TF-IDF

The word features are unigrams and bigrams. The character features are 2 to 4 grams inside word boundaries, which can match the shared stem of inflected Kinyarwanda words. C comes from {0.25, 1, 4, 16}, and the saga solver stops at 220 iterations; the log says when a fit stops there before converging.

In [4]:
def vectorizer(word: bool) -> TfidfVectorizer:
    if word:
        return TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, max_features=80000, sublinear_tf=True)


def fit(estimator, *args):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        estimator.fit(*args)
    return any(issubclass(w.category, ConvergenceWarning) for w in caught)


grid = {}
for name, word in (("word", True), ("char", False)):
    vec = vectorizer(word)
    features = vec.fit_transform(train_text)
    dev_features = vec.transform(dev_text)
    best = None
    for C in CS:
        start = time.time()
        clf = LogisticRegression(C=C, solver="saga", max_iter=220, tol=1e-3, random_state=SEED)
        stopped = fit(clf, features, train_y)
        scores = classification_scores(dev_y, clf.predict(dev_features))
        note = "  stopped at max_iter" if stopped else ""
        print(f"{name:5s} C={C:<5} development macro-F1 {scores['macro_f1']:.4f}  accuracy {scores['accuracy']:.4f}  {time.time() - start:4.0f}s{note}")
        if best is None or scores["macro_f1"] > best[2]["macro_f1"]:
            best = (C, clf, scores)
    C, clf, scores = best
    pipe = Pipeline([("tfidf", vec), ("clf", clf)])
    joblib.dump(pipe, BASELINES / f"{name}.joblib")
    write_jsonl(RESULTS / f"{name}_dev.jsonl", records(dev, align_proba(pipe, dev_text)))
    grid[name] = {"C": C, "dev_macro_f1": scores["macro_f1"]}
    print(f"selected {name} with C={C}\n")

word  C=0.25  development macro-F1 0.3749  accuracy 0.7025     1s


word  C=1.0   development macro-F1 0.5086  accuracy 0.7720     1s


word  C=4.0   development macro-F1 0.5404  accuracy 0.7798     1s


word  C=16.0  development macro-F1 0.5448  accuracy 0.7785     3s


selected word with C=16.0



char  C=0.25  development macro-F1 0.3690  accuracy 0.6920     7s


char  C=1.0   development macro-F1 0.5212  accuracy 0.7628     7s


char  C=4.0   development macro-F1 0.5490  accuracy 0.7746     9s


char  C=16.0  development macro-F1 0.6219  accuracy 0.7851    17s


selected char with C=16.0



## Balanced class weights

The winning character features and C again, with `class_weight="balanced"` so the rare sections count for more in the loss.

In [5]:
char_pipe = joblib.load(BASELINES / "char.joblib")
C = grid["char"]["C"]
balanced = Pipeline([
    ("tfidf", char_pipe.named_steps["tfidf"]),
    ("clf", LogisticRegression(C=C, solver="saga", max_iter=220, tol=1e-3, class_weight="balanced", random_state=SEED)),
])
stopped = fit(balanced, train_text, train_y)
joblib.dump(balanced, BASELINES / "char_balanced.joblib")
dev_records = records(dev, align_proba(balanced, dev_text))
write_jsonl(RESULTS / "char_balanced_dev.jsonl", dev_records)
grid["char_balanced"] = {
    "C": C,
    "class_weight": "balanced",
    "dev_macro_f1": classification_scores(dev_y, [row["pred"] for row in dev_records])["macro_f1"],
}
(RESULTS / "baseline_grid.json").write_text(json.dumps(grid, indent=2) + "\n")
print("char_balanced stopped at max_iter" if stopped else "char_balanced converged")
for name, row in grid.items():
    print(f"{name:14s} C={row['C']:<5} development macro-F1 {row['dev_macro_f1']:.4f}")

char_balanced stopped at max_iter
word           C=16.0  development macro-F1 0.5448
char           C=16.0  development macro-F1 0.6219
char_balanced  C=16.0  development macro-F1 0.6918
